# 🎓 Retail Sales Forecasting: The Master Tutorial (From Scratch)

This notebook provides an end-to-end walkthrough of the project, including data processing, visual exploration (EDA), and the **COMPLETE** implementation of **5 custom machine learning algorithms** built from the ground up without using `scikit-learn` for the core logic.

---

## 1. 📂 Step 1: Library Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add path for src
sys.path.append('..')
from src.data_preprocessing import merge_datasets, clean_data
from src.feature_engineering import extract_features

%matplotlib inline
sns.set(style="whitegrid")

## 2. 🧹 Step 2: Data Preparation

In [ ]:
# Load and prepare a small sample for demonstration
df_raw = merge_datasets('../data')
df = clean_data(df_raw)
df = extract_features(df)

# Evaluation Helper
def evaluate(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred)**2)
    r2 = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    return mae, mse, r2

## 3. 🧠 Step 3: All Custom Algorithms (Full Implementations)

### 3.1 Custom Linear Regression (Gradient Descent)
We use iterative updates based on the error to find the optimal weights.

In [ ]:
class CustomLR:
    def __init__(self, lr=0.001, iters=1000):
        self.lr, self.iters = lr, iters
        self.w, self.b = None, None

    def fit(self, X, y):
        n, f = X.shape
        self.w, self.b = np.zeros(f), 0
        for _ in range(self.iters):
            p = np.dot(X, self.w) + self.b
            self.w -= self.lr * (1/n) * np.dot(X.T, (p - y))
            self.b -= self.lr * (1/n) * np.sum(p - y)

    def predict(self, X): return np.dot(X, self.w) + self.b

### 3.2 Custom Polynomial Regression
We simply transform the input $X$ into $[X, X^2, ...]$ and apply the same Linear Regression.

In [ ]:
class CustomPoly:
    def __init__(self, degree=2, lr=0.000001, iters=1000):
        self.deg, self.lr, self.iters = degree, lr, iters
        self.lr_model = CustomLR(lr=self.lr, iters=self.iters)

    def _transform(self, X):
        out = X.copy()
        for d in range(2, self.deg + 1):
            out = np.hstack([out, np.power(X, d)])
        return out

    def fit(self, X, y):
        X_p = self._transform(X)
        self.lr_model.fit(X_p, y)

    def predict(self, X):
        X_p = self._transform(X)
        return self.lr_model.predict(X_p)

### 3.3 Custom Decision Tree Regressor
Uses recursive splitting based on Variance Reduction (MSE).

In [ ]:
class Node:
    def __init__(self, f_idx=None, th=None, l=None, r=None, v=None):
        self.f_idx, self.th, self.l, self.r, self.v = f_idx, th, l, r, v

class CustomDT:
    def __init__(self, max_d=5):
        self.max_d, self.root = max_d, None

    def _grow(self, X, y, d=0):
        n, f = X.shape
        if d >= self.max_d or n < 2 or np.var(y) == 0: return Node(v=np.mean(y))
        
        best_mse, b_idx, b_th = float("inf"), None, None
        for i in range(f):
            for t in np.unique(X[:, i]):
                l_m, r_m = X[:, i] <= t, X[:, i] > t
                if not np.any(l_m) or not np.any(r_m): continue
                mse = (np.var(y[l_m])*np.sum(l_m) + np.var(y[r_m])*np.sum(r_m)) / n
                if mse < best_mse: best_mse, b_idx, b_th = mse, i, t
        
        if b_idx is None: return Node(v=np.mean(y))
        l = self._grow(X[X[:, b_idx] <= b_th], y[X[:, b_idx] <= b_th], d + 1)
        r = self._grow(X[X[:, b_idx] > b_th], y[X[:, b_idx] > b_th], d + 1)
        return Node(f_idx=b_idx, th=b_th, l=l, r=r)

    def fit(self, X, y): self.root = self._grow(X, y)
    
    def _p_one(self, x, node):
        if node.v is not None: return node.v
        return self._p_one(x, node.l) if x[node.f_idx] <= node.th else self._p_one(x, node.r)

    def predict(self, X): return np.array([self._p_one(x, self.root) for x in X])

### 3.4 Custom Random Forest (Bagging)
Creates a forest of independent trees based on random bootstrapped samples.

In [ ]:
class CustomRF:
    def __init__(self, n_t=5, max_d=5):
        self.n_t, self.max_d, self.trees = n_t, max_d, []

    def fit(self, X, y):
        for _ in range(self.n_t):
            idx = np.random.choice(len(X), len(X), replace=True)
            t = CustomDT(max_d=self.max_d)
            t.fit(X[idx], y[idx])
            self.trees.append(t)

    def predict(self, X):
        return np.mean([t.predict(X) for t in self.trees], axis=0)

### 3.5 Custom Gradient Boosting (XGBoost Core)
Sequentially builds trees to predict the **residuals** of previous predictions.

In [ ]:
class CustomGB:
    def __init__(self, lr=0.1, n_t=5, max_d=3):
        self.lr, self.n_t, self.max_d = lr, n_t, max_d
        self.trees, self.init_v = [], 0

    def fit(self, X, y):
        self.init_v = np.mean(y)
        curr_p = np.full(y.shape, self.init_v)
        for _ in range(self.n_t):
            res = y - curr_p
            t = CustomDT(max_d=self.max_d)
            t.fit(X, res)
            curr_p += self.lr * t.predict(X)
            self.trees.append(t)

    def predict(self, X):
        y_p = np.full(X.shape[0], self.init_v)
        for t in self.trees: y_p += self.lr * t.predict(X)
        return y_p

---